In [ ]:
import torch
torch.cuda.is_available()


True

In [ ]:
!pip install -U transformers datasets torch scikit-learn pandas


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 126.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 140.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 123.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 14.2 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
  Attempting uninstall: datasets
  

In [ ]:
import kagglehub
path = kagglehub.dataset_download("surajkarakulath/labelled-corpus-political-bias-hugging-face")

100%|██████████| 47.8M/47.8M [00:04<00:00, 12.0MB/s]

Extracting files...


In [ ]:
print(path)

/root/.cache/kagglehub/datasets/surajkarakulath/labelled-corpus-political-bias-hugging-face/versions/1


In [ ]:
import os
print(path)
os.listdir(path)
os.listdir(f"{path}/Right Data/Right Data")[:5]

/root/.cache/kagglehub/datasets/surajkarakulath/labelled-corpus-political-bias-hugging-face/versions/1


['1294_0.txt', '4239_0.txt', '390_0 (2).txt', '725_0.txt', '3469_1.txt']

In [ ]:
import os
import pandas as pd

data = []

for label in ["Center Data", "Right Data", "Left Data"]:

    folder = os.path.join(path, label,label)

    for file in os.listdir(folder):

        if file.endswith(".txt"):

            with open(os.path.join(folder, file), "r", encoding="utf-8") as f:
                text = f.read()

            data.append({
                "text": text,
                "label": label
            })

df = pd.DataFrame(data)

df.to_csv("news_bias_dataset.csv", index=False)

print("CSV created successfully")

CSV created successfully


In [ ]:
df["label"] = df["label"].replace({
    "Left Data": "left",
    "Right Data": "right",
    "Center Data": "center"
})

In [ ]:
df.head()

,text,label
0,...,center
1,Transportation Secretary Elaine ChaoElaine Cha...,center
2,The left of the Democratic Party is pinning it...,center
3,...,center
4,(Reuters) - U.S. president-elect Joe Biden nam...,center


In [ ]:
df["label"].value_counts()

label
left      7803
right     5563
center    3996
Name: count, dtype: int64

In [ ]:
import pandas as pd

min_size = df["label"].value_counts().min()

balanced_df = (
    df.groupby("label")
    .sample(min_size, random_state=42)
)

balanced_df["label"].value_counts()


label
center    3996
left      3996
right     3996
Name: count, dtype: int64

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    balanced_df,
    test_size=0.2,
    random_state=42,
    stratify=balanced_df["label"]
)

In [ ]:
from transformers import RobertaTokenizer
import torch
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

train_encodings= tokenizer(
    train_df["text"].tolist(),
    truncation=True,
    padding=True,
    max_length=512
)

test_encodings= tokenizer(
    test_df["text"].tolist(),
    truncation=True,
    padding=True,
    max_length=512
)

class NewsDataset(torch.utils.data.Dataset):
  def __init__(self,encodings,labels):
    self.encodings=encodings
    self.labels=labels
  def __getitem__(self,idx):
    item={key:torch.tensor(val[idx]) for key,val in self.encodings.items()}
    item["labels"]=torch.tensor(self.labels[idx])
    return item
  def __len__(self):
    return len(self.labels)


label_map={
    "left":0,
    "center":1,
    "right":2

}


train_df["label"] = train_df["label"].map(label_map)
test_df["label"] = test_df["label"].map(label_map)

train_dataset = NewsDataset(
    train_encodings,
    train_df["label"].tolist()
)

test_dataset = NewsDataset(
    test_encodings,
    test_df["label"].tolist()
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
from transformers import RobertaForSequenceClassification,TrainingArguments,Trainer

model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=3
)

training_args=TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    logging_steps=100
)

trainer =Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

trainer.train()

model.save_pretrained("bias_model")
tokenizer.save_pretrained("bias_model")





config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Epoch,Training Loss,Validation Loss
1,0.379495,0.317327
2,0.263382,0.306533
3,0.186700,0.337740


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

RuntimeError: on_train_begin must be called before on_evaluate

In [ ]:
!zip -r bias_model.zip bias_model

  adding: bias_model/ (stored 0%)
  adding: bias_model/model.safetensors (deflated 9%)
  adding: bias_model/tokenizer_config.json (deflated 50%)
  adding: bias_model/tokenizer.json (deflated 82%)
  adding: bias_model/config.json (deflated 53%)


In [ ]:
import zipfile


with zipfile.ZipFile("bias_model.zip", "r") as z:
    z.extractall("bias_model")

In [ ]:
from transformers import RobertaForSequenceClassification, RobertaTokenizer, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, classification_report, f1_score
import numpy as np


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions, average="weighted")
    }


model = RobertaForSequenceClassification.from_pretrained("bias_model")
tokenizer = RobertaTokenizer.from_pretrained("bias_model")


training_args = TrainingArguments(
    output_dir="./results",
    per_device_eval_batch_size=8,
)

trainer = Trainer(
    model=model,
    args=training_args,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.callback_handler.on_train_begin(training_args, trainer.state, trainer.control)
results = trainer.evaluate(eval_dataset=test_dataset)
print(results)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Step,Training Loss,Validation Loss,Model Preparation Time,Accuracy,F1
0,No log,0.337740,0.002700,0.931193,0.931129


{'eval_loss': 0.33774039149284363, 'eval_model_preparation_time': 0.0027, 'eval_accuracy': 0.9311926605504587, 'eval_f1': 0.9311291990948375}


In [ ]:
from sklearn.metrics import classification_report
import numpy as np

predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=-1)

print(classification_report(
    predictions.label_ids,
    preds,
    target_names=["Left", "Center", "Right"]
))

Step,Training Loss,Validation Loss,Model Preparation Time,Accuracy,F1
0,No log,0.337740,0.002700,0.931193,0.931129


              precision    recall  f1-score   support

        Left       0.92      0.92      0.92       799
      Center       0.93      0.91      0.92       799
       Right       0.94      0.96      0.95       800

    accuracy                           0.93      2398
   macro avg       0.93      0.93      0.93      2398
weighted avg       0.93      0.93      0.93      2398

